In [1]:
# # SLDCRB–most-informative gap counterexample
#
# Three parameters, rank-three state on C^10.
#
# C_MI(1) > C_S(1) = 54/25
# C_MI(2) = C_S(2) = 27/25
#
# The two-copy equality is certified by ten rational positive-definite 9x9 matrices Q_s.
#
import numpy as np
import sympy as sp
from itertools import product

np.set_printoptions(precision=10, suppress=True)

In [2]:
# ## 1. Construct the quantum statistical model
#
# S = (I_3/3) direct_sum 0_7.
#
# The SLDs have the block form
#
# L_i = [[0, B_i],
#        [B_i.T, 0]].
#
# The tangent matrices are
#
# dS_i = (S L_i + L_i S)/2.


# %%
B0 = np.zeros((3, 7))
B0[0, 0] = 1
B0[1, 1] = 1 / np.sqrt(2)
B0[2, 5] = np.sqrt(2)
B0[2, 6] = np.sqrt(2 / 3)

B1 = np.zeros((3, 7))
B1[0, 1] = np.sqrt(2)
B1[0, 2] = np.sqrt(2 / 3)
B1[1, 0] = 1
B1[2, 3] = 1 / np.sqrt(2)

B2 = np.zeros((3, 7))
B2[0, 5] = 1 / np.sqrt(2)
B2[1, 3] = np.sqrt(2)
B2[1, 4] = np.sqrt(2 / 3)
B2[2, 0] = 1

B = [B0, B1, B2]

S = np.diag([1/3] * 3 + [0] * 7)

L = [
    np.block([
        [np.zeros((3, 3)), b],
        [b.T, np.zeros((7, 7))]
    ])
    for b in B
]

dS = [(S @ Li + Li @ S) / 2 for Li in L]

assert np.isclose(np.trace(S), 1)
assert np.linalg.matrix_rank(S) == 3
assert all(np.allclose(d, d.T) for d in dS)


In [3]:
# ## 2. Calculate the SLD quantum Fisher information
# %%
J = np.array([
    [
        np.trace(
            S @ (L[i] @ L[j] + L[j] @ L[i]) / 2
        )
        for j in range(3)
    ]
    for i in range(3)
])

CS = np.trace(np.linalg.inv(J))

assert np.allclose(J, (25/18) * np.eye(3))
assert np.isclose(CS, 54/25)

print("QFI matrix:")
print(J)
print("C_S(1) =", CS)

QFI matrix:
[[1.3888888889 0.           0.          ]
 [0.           1.3888888889 0.          ]
 [0.           0.           1.3888888889]]
C_S(1) = 2.16


In [4]:
# ## 3. Prove single-copy nonattainability
#
# Define H_ij = B_i B_j.T.
#
# SLDCRB attainability requires H to be separable
# across the parameter-support bipartition.
#
# Any separable positive matrix satisfies the
# realignment criterion
#
# ||R(H)||_1 <= Tr(H).
#
# We find a strict violation.


# %%
H_float = np.vstack(B) @ np.vstack(B).T

R = (
    H_float
    .reshape(3, 3, 3, 3)
    .transpose(0, 2, 1, 3)
    .reshape(9, 9)
)

svals = np.linalg.svd(R, compute_uv=False)

expected = np.array([
    25/6,
    2, 2, 2,
    np.sqrt(139)/6,
    np.sqrt(139)/6,
    0, 0, 0
])

assert np.allclose(svals, expected)

violation = np.sum(svals) - np.trace(H_float)

assert np.isclose(
    violation,
    (np.sqrt(139) - 7)/3
)

assert violation > 0

print("Realignment singular values:", svals)
print("Realignment violation:", violation)
print("Therefore C_MI(1) > C_S(1) = 54/25")

Realignment singular values: [4.1666666667 2.           2.           2.           1.9649710204
 1.9649710204 0.           0.           0.          ]
Realignment violation: 1.596608707517202
Therefore C_MI(1) > C_S(1) = 54/25


In [5]:
# ## 4. Construct the exact two-copy Gram matrix
#
# The additive two-copy SLDs have support-kernel
# Gram blocks
#
# K_ij = H_ij tensor I_3 + I_3 tensor H_ij.
#
# We construct K exactly using SymPy rational
# arithmetic.


# %%
H = sp.Matrix([
    [1,0,0,0,1,0,0,0,1],
    [0,sp.Rational(1,2),0,1,0,0,0,0,0],
    [0,0,sp.Rational(8,3),0,0,0,1,0,0],
    [0,1,0,sp.Rational(8,3),0,0,0,0,0],
    [1,0,0,0,1,0,0,0,1],
    [0,0,0,0,0,sp.Rational(1,2),0,1,0],
    [0,0,1,0,0,0,sp.Rational(1,2),0,0],
    [0,0,0,0,0,1,0,sp.Rational(8,3),0],
    [1,0,0,0,1,0,0,0,1]
])

assert np.allclose(
    np.asarray(H, dtype=float),
    H_float
)

K = sp.zeros(27)

for i in range(3):
    for j in range(3):

        Hij = H[3*i:3*i+3, 3*j:3*j+3]

        Kij = (
            sp.kronecker_product(Hij, sp.eye(3))
            + sp.kronecker_product(sp.eye(3), Hij)
        )

        K[
            9*i:9*i+9,
            9*j:9*j+9
        ] = Kij

assert K == K.T

print("Two-copy Gram matrix constructed exactly.")
print("Dimensions:", K.shape)



Two-copy Gram matrix constructed exactly.
Dimensions: (27, 27)


In [6]:
# ## 5. Exact rational separability certificate
#
# Define the signed cyclic group
#
# G = {C^k diag(1,e1,e2)}
#
# where k=0,1,2 and e1,e2 = +/-1.
#
# The certificate consists of ten positive-definite
# rational 9x9 matrices Q_s satisfying
#
# K = sum_s sum_g
#     (g t_s)(g t_s)^T tensor
#     (g tensor g) Q_s (g tensor g)^T.
#
# Each term is separable. Consequently K is separable.
#
# The data below encode the complete exact certificate.
# Each row gives the 45 upper-triangular entries of a
# matrix numerator. A sparse set of rational corrections
# ensures the identity is EXACT rather than approximate.


# %%
C = sp.Matrix([
    [0,0,1],
    [1,0,0],
    [0,1,0]
])

G = [
    C**k * sp.diag(1, e1, e2)
    for k in range(3)
    for e1,e2 in product((-1,1), repeat=2)
]

assert len(G) == 12

seeds = [
    (1,0,0),
    (1,1,0),
    (1,1,1),
    (1,2,0),
    (1,2,1),
    (1,2,2),
    (1,3,0),
    (1,3,1),
    (1,3,2),
    (1,2,3)
]

DEN = 600000

UPPER_TRIANGLE_INTEGERS = """
47283 0 0 0 -21532 0 0 0 -132499 1321 0 -1173 0 0 0 0 0 31934 0 0 0 -31779 0 0 1321 0 0 0 0 0 9989 0 0 0 60523 2282 0 -501 0 31934 0 0 2282 0 372621
150 1 0 1 -1 0 0 0 -1 1364 0 -1212 4 0 0 0 0 76691 0 0 38622 -76535 -38623 0 1364 4 0 0 0 0 153 0 0 0 -2 20983 -38623 -20834 0 76691 38622 0 20983 0 154
154 2 5 2 0 1 5 1 0 1355 111 -1201 5 -108 -108 111 1 1355 -108 1 111 -1201 -108 2 1355 5 111 111 -108 1 154 2 1 2 0 1355 -108 -1201 5 1355 111 2 1355 5 154
150 2 0 2 -1 0 0 0 -1 1357 0 -1205 4 0 0 0 -1 1365 0 0 139 -1210 -140 0 1357 4 0 0 0 -1 148 0 0 0 -3 1012 -140 -868 0 1365 139 0 1012 0 150
46583 21265 25047 21265 5159 2230 25047 2230 -1980 10300 11488 9327 2367 1003 11450 1039 -907 14232 11450 2783 1234 12935 1172 -1067 10300 2367 1039 11488 1003 -907 722 248 2783 248 -222 1088 1172 -728 -93 14232 1234 -1067 1088 -93 234
150 1 2 1 -1 0 2 0 -2 1327 182 -1174 4 -73 -182 74 0 1063 -182 0 5 -918 -5 0 1327 4 74 182 -73 0 148 1 0 1 -2 1055 -5 -907 4 1063 5 0 1055 4 149
150 2 0 2 -2 0 0 0 -1 1316 0 -1166 2 0 0 0 0 1168 0 0 56 -1013 -57 0 1316 2 0 0 0 0 144 0 0 0 -2 938 -57 -796 0 1168 56 0 938 0 147
155 2 5 2 -3 0 5 0 0 1434 266 -1285 2 19 -266 -18 0 1179 -266 -1 35 -1026 -35 1 1434 2 -18 266 19 0 145 0 -1 0 -2 934 -35 -791 1 1179 35 1 933 1 147
149 1 2 1 -2 0 2 0 -1 10550 5507 -10400 2 -207 -5507 207 0 4103 -5507 0 -107 -3958 107 0 10550 2 207 5507 -207 0 144 0 0 0 -2 930 107 -787 3 4103 -107 0 930 3 147
1815 -5279 58 -5279 -6274 -5270 58 -5270 -4240 17733 -117 15866 19874 16619 -244 16763 13419 946 -244 -215 -189 -800 -171 -145 17733 19874 16763 -117 16619 13419 23757 19833 -215 19833 15943 17707 -171 15758 13397 946 -189 -145 17707 13397 10915
"""

raw = [
    list(map(int, line.split()))
    for line in UPPER_TRIANGLE_INTEGERS.strip().splitlines()
]

assert len(raw) == 10
assert all(len(row) == 45 for row in raw)

flat = [v for row in raw for v in row]

# Sparse exact corrections to the numerator.
# Indices refer to the flattened 10x45 array.
corrections = {
    278:"-12/43",
    291:"-5/43",
    309:"26/43",
    312:"18/43",
    314:"18/43",
    360:"-213/688",
    364:"787/688",
    365:"-1/3",
    367:"-1/3",
    369:"-59/172",
    370:"1/6",
    371:"-189/344",
    374:"1/2",
    377:"217/688",
    378:"1/2",
    387:"1/6",
    391:"1/2",
    393:"1/2",
    398:"-1/3",
    403:"-1/3",
    413:"-11/688",
    426:"21/344",
    444:"-89/172",
    447:"-177/688",
    449:"253/688"
}

for idx, delta in corrections.items():
    flat[idx] = sp.Integer(flat[idx]) + sp.Rational(delta)

pairs = [
    (i,j)
    for i in range(9)
    for j in range(i,9)
]

Qs = []

for s in range(10):

    Q = sp.zeros(9)

    for (i,j), num in zip(
        pairs, flat[45*s:45*(s+1)]
    ):
        Q[i,j] = Q[j,i] = sp.Rational(num, DEN)

    Qs.append(Q)

print("Constructed", len(Qs), "exact rational matrices.")



Constructed 10 exact rational matrices.


In [9]:
Qs[1]

Matrix([
[   1/4000,   1/600000,             0,   1/600000, -1/600000,             0,             0,             0, -1/600000],
[ 1/600000, 341/150000,             0, -101/50000,  1/150000,             0,             0,             0,         0],
[        0,          0,  76691/600000,          0,         0,   6437/100000, -15307/120000, -38623/600000,         0],
[ 1/600000, -101/50000,             0, 341/150000,  1/150000,             0,             0,             0,         0],
[-1/600000,   1/150000,             0,   1/150000, 51/200000,             0,             0,             0, -1/300000],
[        0,          0,   6437/100000,          0,         0,  20983/600000, -38623/600000, -10417/300000,         0],
[        0,          0, -15307/120000,          0,         0, -38623/600000,  76691/600000,   6437/100000,         0],
[        0,          0, -38623/600000,          0,         0, -10417/300000,   6437/100000,  20983/600000,         0],
[-1/600000,          0,             0, 

In [7]:
# ## 6. Verify positive definiteness exactly
#
# For each Q_s, calculate its rational LDL^T decomposition.
#
# If all diagonal elements of D are strictly positive,
# then Q_s is strictly positive definite.
#
# This is an exact proof, not a numerical eigenvalue test.


# %%
for s,Q in enumerate(Qs,1):

    LDL_L, LDL_D = Q.LDLdecomposition(
        hermitian=True
    )

    assert all(
        LDL_D[i,i] > 0
        for i in range(9)
    ), f"Q_{s} is not positive definite"

print("All ten Q_s are positive definite (exact).")

All ten Q_s are positive definite (exact).


In [10]:
# ## 7. Verify the exact separability identity
#
# Check that the orbit sum equals K entry by entry.
#
# SymPy checks the identity over rational arithmetic.


# %%
check = sp.zeros(27)

for t,Q in zip(seeds,Qs):

    t = sp.Matrix(t)

    for g in G:

        gt = g @ t

        gg = sp.kronecker_product(g,g)

        check += sp.kronecker_product(
            gt * gt.T,
            gg * Q * gg.T
        )

assert check == K

print("Exact orbit identity verified.")
print("Residual is exactly zero.")

print(
    "C_MI(2) = C_S(2) =",
    sp.Rational(27,25)
)

# Optional floating-point diagnostic.
min_eigenvalue = min(
    np.linalg.eigvalsh(
        np.asarray(Q,dtype=float)
    ).min()
    for Q in Qs
)

print("Smallest eigenvalue:", min_eigenvalue)

Exact orbit identity verified.
Residual is exactly zero.
C_MI(2) = C_S(2) = 27/25
Smallest eigenvalue: 0.00023298133764419374


In [11]:
# ## 8. Export the certificate
#
# The manuscript refers to a text file containing the
# ten rational Q_s. Generate it here.


# %%
with open("sld_mi_counterexample_Q.txt", "w") as file:

    for s,Q in enumerate(Qs,1):

        file.write(f"Q_{s}\n")

        for i in range(9):

            file.write(
                " ".join(
                    str(Q[i,j])
                    for j in range(9)
                ) + "\n"
            )

        file.write("\n")

print("Wrote sld_mi_counterexample_Q.txt")

Wrote sld_mi_counterexample_Q.txt
